# Flight Delay Prediction - Feature Engineering

**Production-Ready Feature Engineering Pipeline**

This notebook implements proper feature engineering for flight delay prediction:
- ✅ No data leakage
- ✅ Only booking-time features
- ✅ Proper train-test separation
- ✅ Historical features calculated on training data only

## 1. Setup & Configuration

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

# Configuration
DATA_PATH = '/Users/nikitha/Documents/flight-delay-prediction-ice/data/Flight Delay Dataset.csv'
PREPROCESS_DATA = '/Users/nikitha/Documents/flight-delay-prediction-ice/data/'
OUTPUT_PATH = '/Users/nikitha/Documents/flight-delay-prediction-ice/outputs/results/'
TEST_SIZE = 0.2
RANDOM_STATE = 42
DELAY_THRESHOLD = 15  # minutes

### 1.1 Data Overview

In [27]:
# Load data
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
print(f"Total Records: {df.shape[0]:,}")
print(f"Total Features: {df.shape[1]}")

Dataset Shape: (148052, 19)
Total Records: 148,052
Total Features: 19


In [28]:
print("\nDataset Info:")
df.info()


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148052 entries, 0 to 148051
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   YEAR              148052 non-null  int64  
 1   QUARTER           148052 non-null  int64  
 2   MONTH             148052 non-null  int64  
 3   DAY_OF_MONTH      148052 non-null  int64  
 4   DAY_OF_WEEK       148052 non-null  int64  
 5   FL_DATE           148052 non-null  object 
 6   UNIQUE_CARRIER    148052 non-null  object 
 7   ORIGIN            148052 non-null  object 
 8   ORIGIN_CITY_NAME  148052 non-null  object 
 9   ORIGIN_STATE_ABR  148052 non-null  object 
 10  DEST              148052 non-null  object 
 11  DEST_CITY_NAME    148052 non-null  object 
 12  DEST_STATE_ABR    148052 non-null  object 
 13  DEP_TIME          148052 non-null  int64  
 14  ARR_TIME          148052 non-null  int64  
 15  ARR_DELAY         147985 non-null  float64
 16  AIR_T

In [29]:
print("\nStatistical Summary:")
df.describe()


Statistical Summary:


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,DEP_TIME,ARR_TIME,ARR_DELAY,AIR_TIME,DISTANCE,DISTANCE_GROUP
count,148052.000000,148052.000000,148052.000000,148052.000000,148052.000000,148052.000000,148052.000000,147985.00000,148052.000000,148052.000000,148052.000000
mean,2017.343123,2.462142,6.372639,15.775390,3.898786,1344.032347,1477.730905,1.87642,93.435084,657.945033,3.106962
std,0.474754,1.103437,3.380154,8.773131,1.987761,496.274797,512.544797,44.00414,55.380997,469.278191,1.818277
min,2017.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-173.00000,15.000000,83.000000,1.000000
25%,2017.000000,1.000000,3.000000,8.000000,2.000000,939.000000,1103.000000,-15.00000,60.000000,373.000000,2.000000
50%,2017.000000,2.000000,6.000000,16.000000,4.000000,1335.000000,1504.000000,-7.00000,83.000000,566.000000,3.000000
75%,2018.000000,3.000000,9.000000,23.000000,6.000000,1738.250000,1911.000000,4.00000,106.000000,746.000000,3.000000
max,2018.000000,4.000000,12.000000,31.000000,7.000000,2400.000000,2400.000000,1455.00000,631.000000,4502.000000,11.000000


In [30]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)

print("\nMissing Values:")
print(missing_df[missing_df['Percentage'] > 0])


Missing Values:
           Missing_Count  Percentage
ARR_DELAY             67    0.045254


## 2. Creating Target Variable

In [31]:
# Create binary target: delayed if ARR_DELAY > 15 minutes (industry standard)
df['IS_DELAYED'] = (df['ARR_DELAY'] > 15).astype(int)

print("Delay Statistics:")
print(df['ARR_DELAY'].describe())

print(f"\nEarly Arrivals (negative delay): {(df['ARR_DELAY'] < 0).sum():,} ({(df['ARR_DELAY'] < 0).sum()/len(df)*100:.1f}%)")
print(f"On-time (0-15 min): {((df['ARR_DELAY'] >= 0) & (df['ARR_DELAY'] <= 15)).sum():,} ({((df['ARR_DELAY'] >= 0) & (df['ARR_DELAY'] <= 15)).sum()/len(df)*100:.1f}%)")
print(f"Delayed (>15 min): {(df['ARR_DELAY'] > 15).sum():,} ({(df['ARR_DELAY'] > 15).sum()/len(df)*100:.1f}%)")

print(f"\nTarget Variable: IS_DELAYED")
print(f"  Delayed Flights: {df['IS_DELAYED'].sum():,} ({df['IS_DELAYED'].mean()*100:.1f}%)")
print(f"  On-time Flights: {(1-df['IS_DELAYED']).sum():,} ({(1-df['IS_DELAYED'].mean())*100:.1f}%)")

Delay Statistics:
count    147985.00000
mean          1.87642
std          44.00414
min        -173.00000
25%         -15.00000
50%          -7.00000
75%           4.00000
max        1455.00000
Name: ARR_DELAY, dtype: float64

Early Arrivals (negative delay): 100,375 (67.8%)
On-time (0-15 min): 26,298 (17.8%)
Delayed (>15 min): 21,312 (14.4%)

Target Variable: IS_DELAYED
  Delayed Flights: 21,312 (14.4%)
  On-time Flights: 126,740 (85.6%)


In [32]:
# Create target variable
df['IS_DELAYED'] = (df['ARR_DELAY'] > DELAY_THRESHOLD).astype(int)

# Convert date column
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

print(f"Dataset: {df.shape[0]:,} flights")
print(f"Delay rate: {df['IS_DELAYED'].mean()*100:.2f}%")

Dataset: 148,052 flights
Delay rate: 14.39%


## 3. Feature Engineering

### 3.1 Temporal Features (from scheduled date)

In [33]:
# Basic temporal features
df['YEAR'] = df['FL_DATE'].dt.year
df['MONTH'] = df['FL_DATE'].dt.month
df['DAY_OF_MONTH'] = df['FL_DATE'].dt.day
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek + 1

# Day-based binary features
df['IS_WEEKEND'] = df['DAY_OF_WEEK'].isin([6, 7]).astype(int)
df['IS_MONDAY'] = (df['DAY_OF_WEEK'] == 1).astype(int)
df['IS_FRIDAY'] = (df['DAY_OF_WEEK'] == 5).astype(int)
df['IS_SUNDAY'] = (df['DAY_OF_WEEK'] == 7).astype(int)
df['IS_BUSINESS_DAY'] = df['DAY_OF_WEEK'].isin([2, 3, 4]).astype(int)

# Seasonal features
season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}
df['SEASON'] = df['MONTH'].map(season_map)

df['IS_SUMMER'] = df['MONTH'].isin([6, 7, 8]).astype(int)
df['IS_WINTER'] = df['MONTH'].isin([12, 1, 2]).astype(int)
df['IS_HOLIDAY_SEASON'] = df['MONTH'].isin([6, 7, 12]).astype(int)

# Quarter features
df['IS_Q1'] = df['MONTH'].isin([1, 2, 3]).astype(int)
df['IS_Q2'] = df['MONTH'].isin([4, 5, 6]).astype(int)
df['IS_Q3'] = df['MONTH'].isin([7, 8, 9]).astype(int)
df['IS_Q4'] = df['MONTH'].isin([10, 11, 12]).astype(int)

### 3.2 Route & Airport Features

In [34]:
# Route identifier
df['ROUTE'] = df['ORIGIN'] + '-' + df['DEST']
df['CARRIER_ROUTE'] = df['UNIQUE_CARRIER'] + '_' + df['ROUTE']

# Route popularity
route_counts = df['ROUTE'].value_counts()
df['ROUTE_POPULARITY'] = df['ROUTE'].map(route_counts)
df['IS_POPULAR_ROUTE'] = (df['ROUTE_POPULARITY'] > route_counts.median()).astype(int)

# Airport traffic volume
origin_traffic = df['ORIGIN'].value_counts()
dest_traffic = df['DEST'].value_counts()
df['ORIGIN_TRAFFIC'] = df['ORIGIN'].map(origin_traffic)
df['DEST_TRAFFIC'] = df['DEST'].map(dest_traffic)

# Hub airports (top 25%)
df['IS_HUB_ORIGIN'] = (df['ORIGIN_TRAFFIC'] > origin_traffic.quantile(0.75)).astype(int)
df['IS_HUB_DEST'] = (df['DEST_TRAFFIC'] > dest_traffic.quantile(0.75)).astype(int)
df['IS_HUB_TO_HUB'] = (df['IS_HUB_ORIGIN'] & df['IS_HUB_DEST']).astype(int)

# Busy airports (top 50%)
df['IS_BUSY_ORIGIN'] = (df['ORIGIN_TRAFFIC'] > origin_traffic.median()).astype(int)
df['IS_BUSY_DEST'] = (df['DEST_TRAFFIC'] > dest_traffic.median()).astype(int)

### 3.3 Carrier Features

In [35]:
# Carrier volume
carrier_volume = df['UNIQUE_CARRIER'].value_counts()
df['CARRIER_VOLUME'] = df['UNIQUE_CARRIER'].map(carrier_volume)

### 3.4 Distance Features

In [36]:
# Distance categories
df['DISTANCE_CAT'] = pd.cut(df['DISTANCE'], 
                             bins=[0, 500, 1000, 1500, 3000],
                             labels=['Short', 'Medium', 'Long', 'Very_Long'])

df['IS_SHORT_HAUL'] = (df['DISTANCE'] <= 500).astype(int)
df['IS_MEDIUM_HAUL'] = ((df['DISTANCE'] > 500) & (df['DISTANCE'] <= 1500)).astype(int)
df['IS_LONG_HAUL'] = (df['DISTANCE'] > 1500).astype(int)

# Normalized distance
df['DISTANCE_NORMALIZED'] = (df['DISTANCE'] - df['DISTANCE'].mean()) / df['DISTANCE'].std()

### 3.5 Interaction Features

In [37]:
# Time-based interactions
df['WEEKEND_SUMMER'] = (df['IS_WEEKEND'] & df['IS_SUMMER']).astype(int)
df['FRIDAY_SUMMER'] = (df['IS_FRIDAY'] & df['IS_SUMMER']).astype(int)
df['MONDAY_WINTER'] = (df['IS_MONDAY'] & df['IS_WINTER']).astype(int)

# Route-based interactions
df['HUB_TO_HUB_SUMMER'] = (df['IS_HUB_TO_HUB'] & df['IS_SUMMER']).astype(int)
df['POPULAR_ROUTE_WEEKEND'] = (df['IS_POPULAR_ROUTE'] & df['IS_WEEKEND']).astype(int)
df['LONG_HAUL_WINTER'] = (df['IS_LONG_HAUL'] & df['IS_WINTER']).astype(int)

In [38]:
df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'UNIQUE_CARRIER', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR',
       'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEP_TIME', 'ARR_TIME',
       'ARR_DELAY', 'AIR_TIME', 'DISTANCE', 'DISTANCE_GROUP', 'IS_DELAYED',
       'IS_WEEKEND', 'IS_MONDAY', 'IS_FRIDAY', 'IS_SUNDAY', 'IS_BUSINESS_DAY',
       'SEASON', 'IS_SUMMER', 'IS_WINTER', 'IS_HOLIDAY_SEASON', 'IS_Q1',
       'IS_Q2', 'IS_Q3', 'IS_Q4', 'ROUTE', 'CARRIER_ROUTE', 'ROUTE_POPULARITY',
       'IS_POPULAR_ROUTE', 'ORIGIN_TRAFFIC', 'DEST_TRAFFIC', 'IS_HUB_ORIGIN',
       'IS_HUB_DEST', 'IS_HUB_TO_HUB', 'IS_BUSY_ORIGIN', 'IS_BUSY_DEST',
       'CARRIER_VOLUME', 'DISTANCE_CAT', 'IS_SHORT_HAUL', 'IS_MEDIUM_HAUL',
       'IS_LONG_HAUL', 'DISTANCE_NORMALIZED', 'WEEKEND_SUMMER',
       'FRIDAY_SUMMER', 'MONDAY_WINTER', 'HUB_TO_HUB_SUMMER',
       'POPULAR_ROUTE_WEEKEND', 'LONG_HAUL_WINTER'],
      dtype='object')

## 4. Define Feature Lists

In [39]:
# Numerical features
numerical_features = [
    'YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK',
    'DISTANCE', 'DISTANCE_GROUP', 'DISTANCE_NORMALIZED',
    'ROUTE_POPULARITY', 'ORIGIN_TRAFFIC', 'DEST_TRAFFIC',
    'CARRIER_VOLUME', 'DEP_TIME', 'FL_DATE'
]

# Binary features
binary_features = [
    'IS_WEEKEND', 'IS_MONDAY', 'IS_FRIDAY', 'IS_SUNDAY', 'IS_BUSINESS_DAY',
    'IS_SUMMER', 'IS_WINTER', 'IS_HOLIDAY_SEASON',
    'IS_Q1', 'IS_Q2', 'IS_Q3', 'IS_Q4',
    'IS_POPULAR_ROUTE', 'IS_HUB_ORIGIN', 'IS_HUB_DEST', 'IS_HUB_TO_HUB',
    'IS_BUSY_ORIGIN', 'IS_BUSY_DEST',
    'IS_SHORT_HAUL', 'IS_MEDIUM_HAUL', 'IS_LONG_HAUL',
    'WEEKEND_SUMMER', 'FRIDAY_SUMMER', 'MONDAY_WINTER',
    'HUB_TO_HUB_SUMMER', 'POPULAR_ROUTE_WEEKEND', 'LONG_HAUL_WINTER'
]

# Categorical features
categorical_features = [
    'UNIQUE_CARRIER', 'ORIGIN', 'DEST',
    'ORIGIN_STATE_ABR', 'DEST_STATE_ABR',
    'SEASON', 'DISTANCE_CAT', 'ROUTE', 'CARRIER_ROUTE'
]

print(f"Total features: {len(numerical_features) + len(binary_features) + len(categorical_features)}")

Total features: 49


## 5. Train-Test Split

**Critical:** Split BEFORE any target-based encoding to prevent data leakage

In [40]:
# Create feature matrix
feature_cols = numerical_features + binary_features + categorical_features
X = df[feature_cols].copy()
y = df['IS_DELAYED']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training: {len(X_train):,} samples ({y_train.mean()*100:.2f}% delayed)")
print(f"Test: {len(X_test):,} samples ({y_test.mean()*100:.2f}% delayed)")

Training: 118,441 samples (14.40% delayed)
Test: 29,611 samples (14.39% delayed)


In [41]:
# --- INSERT THIS AFTER TRAIN-TEST SPLIT ---

# 1. Create a temporary full dataframe to calculate global traffic patterns
# (We use training data to build the "traffic logic" to avoid leakage, but traffic counts are generally observable)
traffic_df = X_train.copy()
traffic_df['DEP_HOUR'] = (traffic_df['DEP_TIME'] // 100).astype(int)

# Count flights departing from specific airport per hour
hourly_traffic = traffic_df.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR']).size().reset_index(name='HOURLY_DEPARTURES')

# Merge this feature back into X_train and X_test
# Note: For X_test, we fill missing matches with 0 (implies low/no traffic seen in training)
def add_traffic_feature(df_in):
    df_temp = df_in.copy()
    df_temp['DEP_HOUR'] = (df_temp['DEP_TIME'] // 100).astype(int)
    df_merged = pd.merge(df_temp, hourly_traffic, on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')
    df_merged['HOURLY_DEPARTURES'] = df_merged['HOURLY_DEPARTURES'].fillna(0)
    return df_merged.drop(columns=['DEP_HOUR'])

X_train = add_traffic_feature(X_train)
X_test = add_traffic_feature(X_test)

print("Added feature: HOURLY_DEPARTURES")

Added feature: HOURLY_DEPARTURES


In [42]:
# 2. Target Encoding for High Cardinality Categoricals
# We calculate the % probability of delay for each Airport and Carrier based ONLY on X_train
# This prevents "data leakage" (cheating by seeing the future)

# Join X_train with y_train temporarily to calculate rates
train_data = X_train.copy()
train_data['IS_DELAYED'] = y_train.values

# Calculate delay rates
origin_risk = train_data.groupby('ORIGIN')['IS_DELAYED'].mean()
dest_risk = train_data.groupby('DEST')['IS_DELAYED'].mean()
carrier_risk = train_data.groupby('UNIQUE_CARRIER')['IS_DELAYED'].mean()

# Map these rates to X_train
X_train['ORIGIN_RISK'] = X_train['ORIGIN'].map(origin_risk)
X_train['DEST_RISK'] = X_train['DEST'].map(dest_risk)
X_train['CARRIER_RISK'] = X_train['UNIQUE_CARRIER'].map(carrier_risk)

# Map these rates to X_test (Using the rates learned from Train!)
X_test['ORIGIN_RISK'] = X_test['ORIGIN'].map(origin_risk).fillna(train_data['IS_DELAYED'].mean())
X_test['DEST_RISK'] = X_test['DEST'].map(dest_risk).fillna(train_data['IS_DELAYED'].mean())
X_test['CARRIER_RISK'] = X_test['UNIQUE_CARRIER'].map(carrier_risk).fillna(train_data['IS_DELAYED'].mean())

# Drop the original string columns now that we have numeric representations
cols_to_drop = ['ORIGIN', 'DEST', 'UNIQUE_CARRIER', 'FL_DATE', 'TAIL_NUM'] # Add others if present
X_train = X_train.drop(columns=[c for c in cols_to_drop if c in X_train.columns])
X_test = X_test.drop(columns=[c for c in cols_to_drop if c in X_test.columns])

print("Added features: ORIGIN_RISK, DEST_RISK, CARRIER_RISK")

Added features: ORIGIN_RISK, DEST_RISK, CARRIER_RISK


In [43]:
df.to_csv(f"{PREPROCESS_DATA}/Data_preprocessing_flight_delay.csv")
print("CSV created!")

CSV created!


## 6. Categorical Encoding

**No Leakage:** Rates calculated on training data only

In [44]:
print("Columns in X_train:", X_train.columns.tolist())

Columns in X_train: ['YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'DISTANCE', 'DISTANCE_GROUP', 'DISTANCE_NORMALIZED', 'ROUTE_POPULARITY', 'ORIGIN_TRAFFIC', 'DEST_TRAFFIC', 'CARRIER_VOLUME', 'DEP_TIME', 'IS_WEEKEND', 'IS_MONDAY', 'IS_FRIDAY', 'IS_SUNDAY', 'IS_BUSINESS_DAY', 'IS_SUMMER', 'IS_WINTER', 'IS_HOLIDAY_SEASON', 'IS_Q1', 'IS_Q2', 'IS_Q3', 'IS_Q4', 'IS_POPULAR_ROUTE', 'IS_HUB_ORIGIN', 'IS_HUB_DEST', 'IS_HUB_TO_HUB', 'IS_BUSY_ORIGIN', 'IS_BUSY_DEST', 'IS_SHORT_HAUL', 'IS_MEDIUM_HAUL', 'IS_LONG_HAUL', 'WEEKEND_SUMMER', 'FRIDAY_SUMMER', 'MONDAY_WINTER', 'HUB_TO_HUB_SUMMER', 'POPULAR_ROUTE_WEEKEND', 'LONG_HAUL_WINTER', 'ORIGIN_STATE_ABR', 'DEST_STATE_ABR', 'SEASON', 'DISTANCE_CAT', 'ROUTE', 'CARRIER_ROUTE', 'HOURLY_DEPARTURES', 'ORIGIN_RISK', 'DEST_RISK', 'CARRIER_RISK']


In [45]:
# Check which categorical features actually exist in X_train
existing_categorical = [col for col in categorical_features if col in X_train.columns]
missing_categorical = [col for col in categorical_features if col not in X_train.columns]

# Ordinal encoding for features with natural order
ordinal_features = ['SEASON', 'DISTANCE_CAT']
ordinal_features = [col for col in ordinal_features if col in X_train.columns]

for col in ordinal_features:
    le = LabelEncoder()
    X_train[f'{col}_ENCODED'] = le.fit_transform(X_train[col].astype(str))
    
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    X_test[f'{col}_ENCODED'] = X_test[col].astype(str).map(mapping).fillna(-1).astype(int)

print(f"Ordinal encoding complete for: {ordinal_features}")

# Target encoding for high-cardinality features
target_encoding_features = ['UNIQUE_CARRIER', 'ORIGIN', 'DEST', 
                            'ORIGIN_STATE_ABR', 'DEST_STATE_ABR']

# Only encode features that actually exist
target_encoding_features = [col for col in target_encoding_features if col in X_train.columns]

if target_encoding_features:
    for col in target_encoding_features:
        train_temp = X_train.copy()
        train_temp['target'] = y_train.values
        target_mean = train_temp.groupby(col)['target'].mean()
        overall_mean = y_train.mean()
        
        X_train[f'{col}_TARGET_ENC'] = X_train[col].map(target_mean).fillna(overall_mean)
        X_test[f'{col}_TARGET_ENC'] = X_test[col].map(target_mean).fillna(overall_mean)
    
    print(f"Target encoding complete for: {target_encoding_features}")
else:
    print("No target encoding needed - features already encoded or don't exist")

# Drop original categorical columns BUT KEEP ROUTE and CARRIER_ROUTE
cols_to_drop = [col for col in existing_categorical 
                if col in X_train.columns 
                and col not in ['ROUTE', 'CARRIER_ROUTE']]  # ⭐ KEEP THESE!

if cols_to_drop:
    X_train.drop(columns=cols_to_drop, inplace=True)
    X_test.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped categorical columns: {cols_to_drop}")
    print(f"Kept ROUTE and CARRIER_ROUTE for historical features")

print(f"\n Shapes after encoding:")
print(f"   Train: {X_train.shape}")
print(f"   Test: {X_test.shape}")

Ordinal encoding complete for: ['SEASON', 'DISTANCE_CAT']
Target encoding complete for: ['ORIGIN_STATE_ABR', 'DEST_STATE_ABR']
Dropped categorical columns: ['ORIGIN_STATE_ABR', 'DEST_STATE_ABR', 'SEASON', 'DISTANCE_CAT']
Kept ROUTE and CARRIER_ROUTE for historical features

 Shapes after encoding:
   Train: (118441, 49)
   Test: (29611, 49)


## 7. Historical Delay Rate Features

In [46]:
# Verify ROUTE and CARRIER_ROUTE exist
if 'ROUTE' not in X_train.columns or 'CARRIER_ROUTE' not in X_train.columns:
    print("ERROR: ROUTE or CARRIER_ROUTE missing! Cannot create historical features.")
else:
    print("ROUTE and CARRIER_ROUTE found. Creating historical features...")
    
    # Calculate delay rates on TRAINING data only (prevents data leakage)
    route_delay_rate = X_train.groupby('ROUTE').apply(lambda x: y_train.iloc[x.index].mean())
    carrier_route_delay_rate = X_train.groupby('CARRIER_ROUTE').apply(lambda x: y_train.iloc[x.index].mean())
    dow_delay_rate = X_train.groupby('DAY_OF_WEEK').apply(lambda x: y_train.iloc[x.index].mean())
    month_delay_rate = X_train.groupby('MONTH').apply(lambda x: y_train.iloc[x.index].mean())
    
    # Overall mean for unseen values
    overall_mean = y_train.mean()
    
    # Apply to TRAINING set
    X_train['ROUTE_DELAY_RATE'] = X_train['ROUTE'].map(route_delay_rate).fillna(overall_mean)
    X_train['CARRIER_ROUTE_DELAY_RATE'] = X_train['CARRIER_ROUTE'].map(carrier_route_delay_rate).fillna(overall_mean)
    X_train['DOW_DELAY_RATE'] = X_train['DAY_OF_WEEK'].map(dow_delay_rate).fillna(overall_mean)
    X_train['MONTH_DELAY_RATE'] = X_train['MONTH'].map(month_delay_rate).fillna(overall_mean)
    
    # Apply to TEST set (using training statistics)
    X_test['ROUTE_DELAY_RATE'] = X_test['ROUTE'].map(route_delay_rate).fillna(overall_mean)
    X_test['CARRIER_ROUTE_DELAY_RATE'] = X_test['CARRIER_ROUTE'].map(carrier_route_delay_rate).fillna(overall_mean)
    X_test['DOW_DELAY_RATE'] = X_test['DAY_OF_WEEK'].map(dow_delay_rate).fillna(overall_mean)
    X_test['MONTH_DELAY_RATE'] = X_test['MONTH'].map(month_delay_rate).fillna(overall_mean)
    
    print("Historical delay rates added!")
    print(f"New features: {['ROUTE_DELAY_RATE', 'CARRIER_ROUTE_DELAY_RATE', 'DOW_DELAY_RATE', 'MONTH_DELAY_RATE']}")
    
    # NOW drop ROUTE and CARRIER_ROUTE (we've extracted their info)
    X_train.drop(columns=['ROUTE', 'CARRIER_ROUTE'], inplace=True)
    X_test.drop(columns=['ROUTE', 'CARRIER_ROUTE'], inplace=True)
    print(f"Dropped ROUTE and CARRIER_ROUTE (info preserved in delay rates)")
    
    # Verify no missing values
    print(f"\n Missing values in new features:")
    print(f"   Train: {X_train[['ROUTE_DELAY_RATE', 'CARRIER_ROUTE_DELAY_RATE', 'DOW_DELAY_RATE', 'MONTH_DELAY_RATE']].isnull().sum().sum()}")
    print(f"   Test: {X_test[['ROUTE_DELAY_RATE', 'CARRIER_ROUTE_DELAY_RATE', 'DOW_DELAY_RATE', 'MONTH_DELAY_RATE']].isnull().sum().sum()}")

print(f"\n FINAL shapes:")
print(f"   Train: {X_train.shape}")
print(f"   Test: {X_test.shape}")

ROUTE and CARRIER_ROUTE found. Creating historical features...
Historical delay rates added!
New features: ['ROUTE_DELAY_RATE', 'CARRIER_ROUTE_DELAY_RATE', 'DOW_DELAY_RATE', 'MONTH_DELAY_RATE']
Dropped ROUTE and CARRIER_ROUTE (info preserved in delay rates)

 Missing values in new features:
   Train: 0
   Test: 0

 FINAL shapes:
   Train: (118441, 51)
   Test: (29611, 51)


## 8. Handle Missing Values

In [47]:
# Fill missing values with median from training data
if X_train.isnull().sum().sum() > 0 or X_test.isnull().sum().sum() > 0:
    train_median = X_train.median()
    X_train.fillna(train_median, inplace=True)
    X_test.fillna(train_median, inplace=True)
    print("Missing values handled")
else:
    print("No missing values")

No missing values


## 9. Final Verification

In [48]:
# Verify data integrity
assert X_train.isnull().sum().sum() == 0, "NaN values in training data"
assert X_test.isnull().sum().sum() == 0, "NaN values in test data"
assert y_train.isnull().sum() == 0, "NaN values in training target"
assert y_test.isnull().sum() == 0, "NaN values in test target"

print(f"\nFinal Dataset:")
print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")
print(f"Total features: {X_train.shape[1]}")
print(f"\n✅ Feature engineering complete - No data leakage")


Final Dataset:
Training features: (118441, 51)
Test features: (29611, 51)
Total features: 51

✅ Feature engineering complete - No data leakage


## 10. Save Processed Data

In [49]:
# Save to CSV
X_train.to_csv(f'{OUTPUT_PATH}X_train.csv', index=False)
X_test.to_csv(f'{OUTPUT_PATH}X_test.csv', index=False)
y_train.to_csv(f'{OUTPUT_PATH}y_train.csv', index=False)
y_test.to_csv(f'{OUTPUT_PATH}y_test.csv', index=False)

print("Data saved to output directory")

Data saved to output directory
